# C4-classical-ml-practice — Session 3: Pipelines and Honest Validation

*One class session, roughly 85 minutes. Prerequisites: Sessions 1–2
(the `(X, y)` bridge, kNN, standardization with train statistics) and
C1-ml-fundamentals (train-test split, overfitting, accuracy).*

**This session:** you can now build a scaled kNN classifier — this
session is about *trusting the number you report for it*.
We recap C1's evaluation discipline in sklearn form, expose the traps
in the manual scale-then-fit workflow, and meet the two tools that
close them: the **Pipeline**, which welds preprocessing and model into
one object that cannot leak, and **cross-validation**, which replaces
one noisy split with an averaged estimate.
The finale puts them together on the unit's central question — choosing
$k$ *honestly* — worked in full exam register.
The unit-wide **Exam Connections** and **Going Deeper** notes close the
session.

Try every checkpoint yourself before running the verification cells.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804

## 1. Honest Evaluation, Recapped

**The C1 rules, unchanged.**

- Accuracy on rows the model trained on is advertising, not
  measurement — Session 2 showed $k=1$ scoring a meaningless 1.0.
- So: **split** the labeled data, train on one part, measure on the
  held-out part; the split is seeded for reproducibility.
- **Overfitting** shows up as a gap: strong on training rows, weak on
  held-out rows.
- The held-out set is consulted **once**, at the end.
  Every decision made *by looking at it* quietly converts it into
  training material.

**The sklearn verb.**
`train_test_split` performs C1's seeded shuffle-and-slice on `X` and
`y` *together* (same permutation for both — the pairing bug from C1 is
structurally impossible), with `stratify=y` keeping class proportions
equal in both parts:

```python
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)
```

This session's running dataset is sklearn's built-in **wine** table:
178 rows, 13 numeric features on wildly different scales (a perfect
scaling stress test), 3 cultivar classes:

In [ ]:
X, y = load_wine(return_X_y=True)
print("X:", X.shape, "| classes:", np.unique(y), "| counts:", np.bincount(y))
print("feature spreads (std), min and max:",
      X.std(axis=0).min().round(3), "...", X.std(axis=0).max().round(1))

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)
print("train:", X_tr.shape, "test:", X_te.shape)
print("class fractions train:", (np.bincount(y_tr) / len(y_tr)).round(3),
      "test:", (np.bincount(y_te) / len(y_te)).round(3))

### Checkpoint 1

1. Why does `train_test_split` take `X` and `y` in one call instead of
   letting you shuffle each separately?
   Name the C1 bug this design removes.
2. What does `stratify=y` guarantee, and for which kind of dataset
   (think C1's class-imbalance discussion) does it matter most?
3. State the "touch it once" rule for the test set and give one
   concrete example of *indirectly* touching it more than once.

## 2. The Manual Workflow and Its Traps

**The workflow you own so far** (Session 2, by hand):

1. `scaler.fit(X_tr)` — learn $\mu, \sigma$ from training rows only;
2. transform *both* parts with those statistics;
3. fit kNN on the scaled training rows;
4. score on the scaled test rows.

In [ ]:
scaler = StandardScaler().fit(X_tr)                       # stats from train ONLY
Z_tr, Z_te = scaler.transform(X_tr), scaler.transform(X_te)

knn = KNeighborsClassifier(n_neighbors=5).fit(Z_tr, y_tr)
print("scaled kNN, held-out accuracy:", round(knn.score(Z_te, y_te), 4))

raw = KNeighborsClassifier(n_neighbors=5).fit(X_tr, y_tr)
print("raw    kNN, held-out accuracy:", round(raw.score(X_te, y_te), 4))

Four steps, correct — and fragile.
Every step is an opportunity to violate the discipline without any
error message:

- **Trap A (leakage):** fit the scaler on all rows before splitting —
  test rows shape the model's coordinates (Session 2, Pitfall 1).
- **Trap B (mismatched units):** scale the training rows but score on
  raw test rows — the units disagree and the distances are nonsense
  (Session 2, Pitfall 2).

Both bugs *run clean* and return plausible accuracies.
The fix is not more care; it is a structure that makes the bugs
impossible to write.

### Checkpoint 2

1. Write the one incorrect line that commits Trap A, and say which
   rows influenced which statistics.
2. Trap B produces terrible accuracy on the wine data.
   Predict roughly what accuracy, and explain *why* it collapses
   (which feature's raw scale dominates a z-scored table?).
3. Why is "the code ran and printed a number" no evidence of a correct
   evaluation protocol?

## 3. Pipelines: the Whole Recipe as One Object

**Definition.**
A **`Pipeline`** chains named steps — any number of transformers, then
one final model — into a single object that speaks the ordinary
estimator API:

```python
pipe = Pipeline([("scaler", StandardScaler()),
                 ("knn",    KNeighborsClassifier(n_neighbors=5))])
```

- `pipe.fit(X_tr, y_tr)` — fits the scaler **on `X_tr` only**,
  transforms `X_tr` with those statistics, then fits the classifier on
  the result.
- `pipe.predict(Q)` / `pipe.score(X_te, y_te)` — pushes new rows
  through the *already-fitted* scaler (training statistics — never
  refit), then through the classifier.

Read that twice: the fit/transform discipline you enforced by hand in
Section 2 is now the *only behavior the object is capable of*.
Trap A cannot happen because `fit` never sees test rows; Trap B cannot
happen because every prediction path leads through the stored scaler.

In [ ]:
pipe = Pipeline([("scaler", StandardScaler()),
                 ("knn",    KNeighborsClassifier(n_neighbors=5))])

pipe.fit(X_tr, y_tr)
print("pipeline held-out accuracy:", round(pipe.score(X_te, y_te), 4))

# Identical to the careful manual workflow -- verify:
manual_preds = knn.predict(Z_te)
print("agrees with manual workflow on every test row:",
      (pipe.predict(X_te) == manual_preds).all())

# The fitted scaler lives inside, holding TRAIN statistics:
print("inner scaler mean (first 3 features):",
      pipe.named_steps["scaler"].mean_[:3].round(2))

### Checkpoint 3

1. List what happens, in order, inside `pipe.fit(X_tr, y_tr)` — which
   step sees which data, and what does each step store?
2. When `pipe.score(X_te, y_te)` runs, is the scaler refit on `X_te`?
   What happens to `X_te` instead?
3. A pipeline is built with the steps in the wrong order —
   `[("knn", ...), ("scaler", ...)]`.
   Why can this never work?
   (What must every non-final step be able to do?)

## 4. Cross-Validation: Averaging Away Split Luck

**Motivation.**
One split = one number, and that number depends on *which* rows landed
in the test part:

In [ ]:
accs = []
for rs in range(8):                                   # 8 different split seeds
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25,
                                      random_state=rs, stratify=y)
    p = Pipeline([("scaler", StandardScaler()),
                  ("knn", KNeighborsClassifier(n_neighbors=5))]).fit(Xa, ya)
    accs.append(p.score(Xb, yb))
print("same model, eight splits:", np.round(accs, 3))
print("spread:", round(max(accs) - min(accs), 3))

A spread of ~0.07 from split luck alone — comparable to the difference
between a good and a mediocre $k$.
Decisions need a steadier estimate.

**Definition (k-fold cross-validation).**
Split the *training* data into $K$ equal **folds** (here $K = 5$).
For each fold in turn: train on the other $K - 1$ folds, evaluate on
the held-out fold.
That yields $K$ scores from $K$ models; report their mean (and spread).

**The arithmetic** (worth having cold — it is quiz material):
$n$ rows, $K$ folds → each fold has $n/K$ rows; each row is
**validated exactly once** and used for training $K - 1$ times; $K$
models are trained, each on $n(K-1)/K$ rows.

**The verb.**
`cross_val_score(estimator, X_tr, y_tr, cv=5)` does all of it —
cloning the *unfitted* estimator per fold — and returns the 5 scores.
For classifiers it uses stratified folds automatically.
Passing the **pipeline** as the estimator means scaling is refit
inside every fold, on that fold's training portion only — more on why
that matters in Section 6:

In [ ]:
pipe = Pipeline([("scaler", StandardScaler()),
                 ("knn", KNeighborsClassifier(n_neighbors=5))])

scores = cross_val_score(pipe, X_tr, y_tr, cv=5)      # 5 fits, 5 held-out scores
print("fold scores:", scores.round(4))
print("mean:", scores.mean().round(4), "| std:", scores.std().round(4))

Note what cross-validation is *for*: **comparing and choosing**
(models, $k$ values, preprocessing) using training data only.
The untouched test set still waits for the single final measurement.

### Checkpoint 4

1. For 10-fold CV on 120 rows: how many models are trained, on how
   many rows each, and how many times is each row validated?
2. The five fold scores above range from ~0.89 to 1.0.
   Why is their *mean* a steadier estimate than any single
   train/test split of the same data?
3. `cross_val_score` receives the pipeline *unfitted*.
   Why would passing a pre-fitted estimator miss the point?

## 5. Worked Exam-Style Example: Choosing k Honestly

The full model-selection arc, in the register the applied problem uses
(paraphrased): exact identifiers, a pinned procedure, and a **ban with
a zero-points clause**.

> **Task.**
> For the iris dataset (150 rows, 4 features, 3 classes):
> **(a)** split off 25% as a test set
> (`random_state=SEED, stratify=y`);
> **(b)** for each $k$ in `[1, 3, 5, 7, 9, 11]`, compute `cv_means[i]`
> — the mean 5-fold CV accuracy on the *training part* of a
> scaler+kNN pipeline;
> **(c)** set `best_k` to the $k$ with the highest mean, **smallest
> $k$ winning ties**;
> **(d)** refit the pipeline with `best_k` on the full training part
> and report `test_acc` on the held-out rows — the test set's first
> and only use.
> **Constraint (zero points): the test rows must not appear in any
> step before (d).**

**Solution, narrated.**
(a) is Section 1; (b) is a loop over six candidate values, each judged
by `cross_val_score` on training data only; (c) `np.argmax` returns
the *first* maximum, which — with `ks` ascending — implements the
tie-break for free (state this; graders look for it); (d) is the
single sanctioned appearance of the test set.

In [ ]:
X_ir, y_ir = load_iris(return_X_y=True)

# (a) one seeded, stratified split
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    X_ir, y_ir, test_size=0.25, random_state=SEED, stratify=y_ir)

# (b) CV-score each candidate k on the TRAINING part only
ks = np.array([1, 3, 5, 7, 9, 11])
cv_means = np.array([
    cross_val_score(
        Pipeline([("scaler", StandardScaler()),
                  ("knn", KNeighborsClassifier(n_neighbors=int(k)))]),
        Xi_tr, yi_tr, cv=5).mean()
    for k in ks])
print("k        :", ks)
print("cv_means :", cv_means.round(4))

# (c) best k, smallest wins ties (argmax = first maximum, ks ascending)
best_k = int(ks[np.argmax(cv_means)])
print("best_k   :", best_k)

# (d) first and only use of the test rows
final = Pipeline([("scaler", StandardScaler()),
                  ("knn", KNeighborsClassifier(n_neighbors=best_k))])
final.fit(Xi_tr, yi_tr)
test_acc = final.score(Xi_te, yi_te)
print("test_acc :", round(test_acc, 4))

The CV means bunch around 0.92–0.93 with $k = 5$ (tied by $k = 9$,
smaller wins) on top; the final, once-only test evaluation lands
high — on an easy dataset the honest procedure loses nothing.
Grader's-eye view: the deliverables are `cv_means`, `best_k`,
`test_acc` under exactly those names; the instant-zero mistakes are
scoring candidates on the test set, or "re-checking" other $k$ values
on it after (d).

### Checkpoint 5

1. In (b), each candidate's score comes from data the corresponding
   model *did* eventually train on (other folds).
   Why is that acceptable for *choosing*, while touching the test set
   is not?
2. Suppose `cv_means` were `[0.90, 0.94, 0.94, 0.93, 0.94, 0.92]`.
   What is `best_k` under the tie rule, and which NumPy behavior
   delivers it automatically?
3. After (d) you notice $k = 13$ was never tried.
   What is the *honest* way to test it — and what does that cost?

## 6. Why the Pipeline Must Live Inside the CV

**Motivation.**
A tempting "optimization": standardize the whole table once, then
cross-validate a bare kNN on the scaled matrix.
It looks equivalent.
It is not: within each CV round, the held-out fold's rows already
shaped the $\mu, \sigma$ used to scale the training folds — every fold
"knows" statistics of its own validation rows.
That is Trap A smuggled inside the CV loop.

Three protocols on Session 2's sensor data (two informative small-scale
features, one huge-scale noise feature):

In [ ]:
rng = np.random.default_rng(SEED)
ok = np.column_stack([rng.normal(65.0, 1.8, 30), rng.normal(3.0, 0.7, 30),
                      rng.normal(101000.0, 1400.0, 30)])
faulty = np.column_stack([rng.normal(68.2, 1.8, 30), rng.normal(4.6, 0.7, 30),
                          rng.normal(101000.0, 1400.0, 30)])
X_m = np.vstack([ok, faulty])
y_m = np.array([0] * 30 + [1] * 30)

knn5 = KNeighborsClassifier(n_neighbors=5)

acc_raw = cross_val_score(knn5, X_m, y_m, cv=5).mean()           # no scaling
Z_all = StandardScaler().fit_transform(X_m)                       # LEAKY pre-scale
acc_leaky = cross_val_score(knn5, Z_all, y_m, cv=5).mean()
acc_pipe = cross_val_score(
    Pipeline([("scaler", StandardScaler()), ("knn", knn5)]),      # honest
    X_m, y_m, cv=5).mean()

print(f"raw (no scaling)      : {acc_raw:.4f}")    # ~0.50 -- scale distortion
print(f"pre-scaled, leaky     : {acc_leaky:.4f}")  # ~0.87
print(f"pipeline inside CV    : {acc_pipe:.4f}")   # ~0.85 -- the honest number

Reading the three numbers:

- **raw 0.50**: Session 2's distortion — no protocol can rescue the
  wrong representation;
- **leaky 0.867 vs honest 0.85**: the difference is small *here* — a
  scaler leaks only two numbers per column — but the leaky number is
  systematically the flattering one, its bias is unquantified, and
  with richer preprocessing (feature selection, per-class statistics)
  the same wiring error produces wildly optimistic scores.
  The pipeline version is the only one whose meaning — "performance on
  rows that influenced nothing" — is intact.

Rule: **whatever is learned from data must be learned inside the
fold** — and passing a pipeline to `cross_val_score` is that rule in
one line.

### Checkpoint 6

1. Trace the leak: in the pre-scaled protocol, name the exact numbers
   ($\mu_j$? $\sigma_j$? the model's neighbors?) that the validation
   fold's rows influenced before being predicted.
2. Why does the leak stay small for a `StandardScaler` (what — and
   how little — does a scaler actually learn from the data)?
3. State the general rule in your own words, and apply it: a
   preprocessing step drops the features least correlated with the
   label. May it run once on the full table before CV?

## 7. Common Pitfalls III

**Pitfall 1 — contiguous folds on a sorted file.**
Many CSVs arrive sorted by class.
Hand-rolled CV that slices contiguous blocks then validates each block
against the rest can end up validating on classes absent from
training — accuracy 0, or (subtler) folds with distorted class
mixes.
`cross_val_score` stratifies classifier folds automatically, but
*manual* fold code (and lazy `array_split` on unshuffled data) walks
straight into it:

In [ ]:
# iris arrives sorted: rows 0-49 class 0, 50-99 class 1, 100-149 class 2.
X_sorted, y_sorted = load_iris(return_X_y=True)
folds = np.array_split(np.arange(150), 3)          # BROKEN: contiguous thirds

for f in range(3):
    val = folds[f]
    tr = np.concatenate([folds[g] for g in range(3) if g != f])
    acc = KNeighborsClassifier(n_neighbors=5).fit(
        X_sorted[tr], y_sorted[tr]).score(X_sorted[val], y_sorted[val])
    print(f"fold {f}: validating on class {set(y_sorted[val])} -> accuracy {acc}")
print("every fold validates on a class the model never saw: 0.0 across the board")

Fix: shuffle with a seeded permutation before manual folding (or
stratify); then each fold contains all classes.

**Pitfall 2 — choosing k on the test set.**
Sweeping $k$ and reporting the best *test* accuracy converts the test
set into a validation set; the reported maximum is selection luck, not
skill.
Pure-noise features make the inflation visible — no procedure can
genuinely beat 0.5 here, but best-of-ten peeking claims 0.67:

In [ ]:
rng = np.random.default_rng(SEED)
X_noise = rng.normal(0, 1, (40, 4))               # features: pure noise
y_noise = rng.integers(0, 2, 40)                  # labels: coin flips

Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(
    X_noise, y_noise, test_size=12, random_state=SEED, stratify=y_noise)

peeked = [KNeighborsClassifier(n_neighbors=int(k)).fit(Xn_tr, yn_tr)
          .score(Xn_te, yn_te) for k in np.arange(1, 20, 2)]
print("test accuracies while 'tuning' on the test set:", np.round(peeked, 3))
print("reported 'best':", round(max(peeked), 3), " <- selection luck on 12 rows")

Fix: candidates are compared by CV on training data (Section 5); the
test set sees only the single already-chosen model.

**Pitfall 3 — reporting the best fold.**
Fold scores are $K$ noisy estimates of one quantity; the maximum of
$K$ noisy numbers is biased upward, always.
Report mean (and spread), never the best fold:

In [ ]:
scores = cross_val_score(Pipeline([("scaler", StandardScaler()),
                                   ("knn", KNeighborsClassifier(n_neighbors=5))]),
                         X_tr, y_tr, cv=5)
print("fold scores :", scores.round(4))
print("honest      :", round(scores.mean(), 4), "+/-", round(scores.std(), 4))
print("cherry-pick :", round(scores.max(), 4), " <- not an estimate of anything")

**Pitfall 4 — model settings chosen, then split changed.**
Choosing $k$ with one `random_state`, then re-running everything with
a "nicer" split until `test_acc` looks good, is Pitfall 2 in slow
motion — each re-roll is another peek.
Fix: freeze `SEED`, run the protocol once, report what comes out.

### Checkpoint 7

1. A classmate's manual 5-fold CV on a class-sorted CSV reports
   fold accuracies `[0.0, 0.1, 0.9, 0.2, 0.0]`.
   Diagnose the two distinct problems visible in that pattern.
2. In the noise demo, roughly what accuracy *should* an honest
   procedure report, and why does 0.67 appear anyway?
3. Rank by trustworthiness as "the number to report":
   (i) best fold score, (ii) mean CV score of the *chosen* model,
   (iii) the final once-only test accuracy.
   Justify in one sentence each.

## Exam Connections

How this unit's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic table — no verbatim test text):

- The **applied tabular-ML problem is the single largest item on the
  paper**: one integrated arc worth 50 of 300 points (r1-2026),
  demanding exactly this unit's chain — load a CSV, build `(X, y)`,
  scale features, fit a k-nearest-neighbors classifier, and evaluate
  it defensibly (the paper grades with a macro-averaged F1 from C1's
  metric family rather than plain accuracy — same discipline, sterner
  metric).
- Its rules (paraphrased) **allow scikit-learn but restrict the model
  family to k-nearest neighbors** — Sessions 2–3's
  `KNeighborsClassifier` + `StandardScaler` + `Pipeline` +
  `cross_val_score` toolkit is not a subset of what you need; it *is*
  the sanctioned toolkit.
- **Starter code arrives in Session 1's idiom** — read a CSV, select
  feature columns, bridge to NumPy — and the data-wrangling sub-parts
  grade named identifiers with pinned shapes, the register of every
  worked example in this unit.
- Programming sub-parts across the paper carry **explicit API bans
  with zero-points clauses**; here the craft is the *scaling and
  evaluation discipline* — train-statistics only, test touched once —
  because the graders' scenario questions probe exactly those
  protocol points.
- Concept MCs (five options A–E, occasionally numeric answers in
  normal form) draw on this unit's vocabulary: what standardization
  does to a distance, what $K$-fold CV trains and validates, why
  $k = 1$ overfits — p02/p03/p04's register.

Highest-yield hour if the exam were tomorrow: the CSV → `(X, y)`
bridge, one raw-vs-scaled kNN comparison, and the Section 5 honest-$k$
arc, all from memory.

## Going Deeper

Optional forward pointers along the course map — nothing here is
needed for this unit's practice:

- **`C10-competition-craft`**: this unit chose $k$ honestly on data it
  could see; the competition setting adds a *hidden* test set and a
  required prediction-function deliverable.
  The protocol and packaging craft for that setting — C10's
  `prediction-function-contract` and `hidden-test-protocol` topics —
  build directly on Session 3's discipline, and that unit (not this
  one) is where those rules live.
- **`C2-linear-models`**: kNN classifies by *memory* — the whole
  training set is the model.
  The next core unit starts the other family: models that compress
  data into a few learned numbers, opening the door to
  loss functions and, later, training loops.
- **`C8-embeddings`**: "find the nearest rows" returns at scale —
  nearest-neighbor search over thousands of high-dimensional
  embedding vectors, where Session 2's distance machinery meets
  serious engineering.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Rows and labels must be shuffled by the *same* permutation; one
   call on both makes desynchronized shuffling (C1's pairing bug —
   features re-ordered, labels not) impossible to write.
2. Both parts get (as nearly as possible) the same class proportions
   as the full data; it matters most for imbalanced datasets, where an
   unlucky split can leave a rare class nearly absent from one side.
3. The test set is evaluated once, by the final already-chosen model.
   Indirect violation example: sweeping $k$, looking at test accuracy
   for each, and "just" using the best — every look was a use.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. `scaler = StandardScaler().fit(X)` (all rows, before splitting) —
   the test rows helped determine every $\mu_j, \sigma_j$.
2. Near chance (~1/3 for balanced 3-class wine, often worse): the
   z-scored training table has every feature at spread ~1; a raw test
   row's `proline` values (hundreds to ~1700) sit thousands of
   "standard units" away, so distances are pure `proline` noise.
3. Protocol bugs produce no exceptions — the arithmetic is valid, the
   *meaning* of the number is not; only the data-flow (who saw what,
   when) determines whether the number measures generalization.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. (i) `scaler.fit(X_tr)` — sees training rows, stores per-column
   $\mu, \sigma$; (ii) `scaler.transform(X_tr)` — applies them;
   (iii) `knn.fit(Z_tr, y_tr)` — sees the scaled training rows,
   stores them (kNN memorizes).
2. No — `X_te` is pushed through `transform` with the *stored
   training* statistics, then predicted on; `fit` is never called
   during scoring.
3. Every non-final step must be a transformer (have
   `fit`/`transform`); a classifier has no `transform`, and its
   output (labels) is not a feature table a scaler could sensibly
   process — the chain type-checks only as transformers → model.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. 10 models; each trains on $120 \cdot 9/10 = 108$ rows; each row is
   validated exactly once (and trains 9 models).
2. Each fold score is computed on different held-out rows; averaging
   the 5 estimates cancels much of the "which rows landed where" luck
   that a single split bakes into its one number.
3. The point is to measure the whole *recipe* refit from scratch on
   each fold's training portion; a pre-fitted estimator would carry
   information from outside the fold (including its validation rows)
   into every round.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Model selection only needs a *fair comparison* among candidates —
   each candidate is scored on rows held out from its own fit, and all
   candidates are treated identically.
   The final *reported claim* needs more: rows that influenced no
   choice at all — which is exactly what the untouched test set
   provides.
2. `best_k = 3`: the maximum 0.94 first occurs at index 1, and
   `np.argmax` returns the first maximum; with `ks` sorted ascending
   that is the smallest tied $k$.
3. Add $k = 13$ to the CV comparison (training data only), and if it
   wins, refit — but the old `test_acc` is now stale, and honesty
   requires *fresh* held-out data (or accepting the original result):
   the cost of the test set's once-only nature.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Every $\mu_j$ and $\sigma_j$ of the pre-scaled matrix — computed
   from all rows, including the fold's validation rows; through them,
   the coordinates of every training row the model measures distances
   in.
2. A scaler learns just $2d$ numbers (a mean and a spread per
   column), each an average over many rows — one row's influence is
   tiny, so the optimism is small (but nonzero, and unquantified).
3. Rule: any step that learns numbers from data must see only the
   training portion of its fold.
   The correlation-based feature dropper learns *which columns
   survive* from labels — run once on the full table it leaks label
   information from every validation row; it must go inside the
   pipeline.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. (i) Contiguous folding on sorted data: some folds validate on
   classes missing (or nearly missing) from training — the 0.0
   folds; (ii) the huge fold-to-fold spread (0.0 to 0.9) signals the
   folds are not exchangeable samples — the protocol, not the model,
   made the numbers.
2. ~0.5 — the features carry no label information, so no classifier
   genuinely beats coin-flipping; 0.67 is the maximum of ten noisy
   ~0.5 estimates on 12 rows — selection applied to noise.
3. (iii) > (ii) > (i).
   (iii) measured a single pre-chosen model on rows that influenced
   nothing; (ii) is training-data-only and averaged, but the chosen
   model was *selected* to maximize it, so it is mildly optimistic;
   (i) is the maximum of noisy draws — biased upward by
   construction.

</details>